In [13]:
from astropy.io import fits
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import curve_fit
from astropy.modeling import models
from scipy.signal import convolve
from scipy.optimize import minimize
from astropy.convolution import Gaussian1DKernel
import scipy.integrate as integrate
from scipy.optimize import differential_evolution
import pandas as pd
import os
from scipy.integrate import simpson

In [14]:
hdul = fits.open("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/Mrk_544/FORS2.2021-08-21T05_32_16.048/Reduced/Mrk-544_MAPPED_FLUX_SCI_LSS_U1.fits")
hdul.info()
image = hdul[0].data
image_error = hdul[1].data

Filename: C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/Mrk_544/FORS2.2021-08-21T05_32_16.048/Reduced/Mrk-544_MAPPED_FLUX_SCI_LSS_U1.fits
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU     365   (2094, 964)   float32   
  1  IMAGE.ERR     1 ImageHDU        46   (2094, 964)   float32   


In [15]:
image_cut = image[55:155, 0:1890]
image_error_cut = image_error[55:155, 0:1890]

In [16]:
wv = np.arange(4300.74, 4300.74+1890*1.48, 1.48)
J, N = image_cut.data.shape
#x = np.arange(0, N, 1)
y = np.arange(0, J, 1)
arcsec_y = np.arange(-58*0.256, 42*0.256, 0.256)

z = 0.0238
l_HA = 6563 * (1+z)
l_HB = 4861 * (1+z) 
l_NII = 6583 * (1+z)
l_SII = 6717 * (1+z)
l_OIII = 5007 * (1+z)

In [17]:
def calculate_bn(n):
    """ Calcola b_n per il profilo Sérsic """
    return 2*n - 1/3 + 4/(405*n) + 46/(25515*n**2)

def sersic_1d(x, params):
    """ Profilo Sérsic 1D centrato in x_0 """
    I_e, r_e, n, x_0 = params
    r = np.abs(x - x_0)
    b_n = calculate_bn(n)
    return I_e * np.exp(-b_n * ((r / r_e)**(1 / n) - 1))
    
def sersic_1d_convolved(x, params):
    """ Sérsic convoluto con PSF gaussiana """
    profile = sersic_1d(x, params)
    psf = Gaussian1DKernel(stddev = sigma, mode='center')
    conv_profile = convolve(profile, psf, mode='same')
    return conv_profile

def model_convolved(x, params1, params2, params3, params4):
    x_hr = np.linspace(min(x), max(x), 4000)
    profile = (
        sersic_1d(x_hr, params1) +
        sersic_1d(x_hr, params2) +
        sersic_1d(x_hr, params3) +
        sersic_1d(x_hr, params4)
    )
    kernel = Gaussian1DKernel(stddev=(sigma*len(x_hr))/len(x), x_size=len(x_hr), mode='center')   
    conv_profile = convolve(profile, kernel, mode='same')
    conv_profile_out = np.interp(x, x_hr, conv_profile)
    return conv_profile_out

In [18]:
seeing = 3.174
sigma = seeing / 2.35 
x_axes = np.arange(0, 100, 1)

In [19]:
def single_integral(x, params, a, b):
    # x --> array di x 
    # y --> dati sulla y (sersic_1d_conv(x))
    # params --> parametri del profilo 
    # a e b --> estremi di integrazione
    func = sersic_1d_convolved(x, params)
    mask = (x >= a)*(x <= b)
    integral = simpson(y = func[mask], x = x[mask])
    return integral 

def total_integral(x, params1, params2, params3, params4, a, b):
    func = model_convolved(x, params1, params2, params3, params4)
    mask = (x >= a)*(x <= b)
    integral = simpson(y = func[mask], x = x[mask])
    return integral  

def contamination(x, params1, params2, params3, params4, a, b): 
    blue_area = single_integral(x, params1, a, b) 
    green_area = single_integral(x, params2, a, b)
    purple_area = single_integral(x, params3, a, b)
    orange_area = single_integral(x, params4, a, b)
    sum_area = blue_area + green_area +purple_area + orange_area
    total_area = total_integral(x, params1, params2, params3, params4, a, b)
    print(sum_area)
    print(total_area)
    return [(blue_area*100)/sum_area, (green_area*100)/sum_area, (purple_area*100)/sum_area, (orange_area*100)/sum_area]     

## Halpha

In [20]:
mask_HA = (wv > l_HA-8)*(wv < l_HA+8)
image_HA = image_cut[:, mask_HA]
image_error_HA = image_error_cut[:, mask_HA]
radial_profile_HA = np.sum(image_HA, axis = 1)
radial_profile_error_HA = np.sqrt(np.sum(image_error_HA**2, axis = 1))
fit_HA = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/Mrk_544/FORS2.2021-08-21T05_32_16.048/Reduced/Fit_code/Halpha_fit.csv", index_col = 0)

In [21]:
data1 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], 
                                                                fit_HA['Component 4'], fit_HA['Component 1'].iloc[3] -sigma, 
                                                                fit_HA['Component 1'].iloc[3] + sigma)
data2 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], 
                                                                fit_HA['Component 4'], fit_HA['Component 2'].iloc[3] -sigma, 
                                                                fit_HA['Component 2'].iloc[3] + sigma)
data3 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], 
                                                                fit_HA['Component 4'], fit_HA['Component 3'].iloc[3] -sigma, 
                                                                fit_HA['Component 3'].iloc[3] + sigma)
data4 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], 
                                                                fit_HA['Component 4'], fit_HA['Component 4'].iloc[3] -sigma, 
                                                                fit_HA['Component 4'].iloc[3] + sigma)
data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(4)],
                  index=['Blue', 'Green', 'Purple', 'Orange'])
print(df)

df_path = os.path.join('Areas_Halpha_1s.csv')
df.to_csv(df_path, float_format='%.3f')

21.206717870539116
21.60803089898615
4.222077539277411
4.453899157507356
3.882554345718614
4.84395948146155
1.747847412480268
1.7772721080650729
           Peak 1     Peak 2     Peak 3     Peak 4
Blue    95.667476  42.327264   1.661413   1.241271
Green    3.463010  54.104832   1.457132   1.533048
Purple   0.470522   2.222787  92.877292  18.845977
Orange   0.398993   1.345116   4.004163  78.379705


In [22]:
data1 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], 
                                                                fit_HA['Component 4'], fit_HA['Component 1'].iloc[3] -2*sigma, 
                                                                fit_HA['Component 1'].iloc[3] + 2*sigma)
data2 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], 
                                                                fit_HA['Component 4'], fit_HA['Component 2'].iloc[3] -2*sigma, 
                                                                fit_HA['Component 2'].iloc[3] + 2*sigma)
data3 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], 
                                                                fit_HA['Component 4'], fit_HA['Component 3'].iloc[3] -2*sigma, 
                                                                fit_HA['Component 3'].iloc[3] + 2*sigma)
data4 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], 
                                                                fit_HA['Component 4'], fit_HA['Component 4'].iloc[3] -2*sigma, 
                                                                fit_HA['Component 4'].iloc[3] + 2*sigma)
data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(4)],
                  index=['Blue', 'Green', 'Purple', 'Orange'])
print(df)

df_path = os.path.join('Areas_Halpha_2s.csv')
df.to_csv(df_path, float_format='%.3f')

38.872061158848155
39.51462884885679
20.189850062717944
20.963640935556693
16.259093137584156
19.473849672519094
4.031854549347862
4.0868510026456635
           Peak 1     Peak 2     Peak 3     Peak 4
Blue    95.095102  48.465542   2.033291   1.280303
Green    3.952995  47.763590   1.776743   1.596968
Purple   0.516015   2.360299  91.367541  19.516280
Orange   0.435887   1.410569   4.822426  77.606450


In [23]:
data1 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], 
                                                                fit_HA['Component 4'], fit_HA['Component 1'].iloc[3] -3*sigma, 
                                                                fit_HA['Component 1'].iloc[3] + 3*sigma)
data2 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], 
                                                                fit_HA['Component 4'], fit_HA['Component 2'].iloc[3] -3*sigma, 
                                                                fit_HA['Component 2'].iloc[3] + 3*sigma)
data3 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], 
                                                                fit_HA['Component 4'], fit_HA['Component 3'].iloc[3] -3*sigma, 
                                                                fit_HA['Component 3'].iloc[3] + 3*sigma)
data4 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], 
                                                                fit_HA['Component 4'], fit_HA['Component 4'].iloc[3] -3*sigma, 
                                                                fit_HA['Component 4'].iloc[3] + 3*sigma)
data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(4)],
                  index=['Blue', 'Green', 'Purple', 'Orange'])
print(df)

df_path = os.path.join('Areas_Halpha_3s.csv')
df.to_csv(df_path, float_format='%.3f')

57.881013875849206
58.647137125643184
28.037400086208297
28.87315240790268
19.793645086150562
23.1423899315711
5.350784508817558
5.407063790480841
           Peak 1     Peak 2     Peak 3     Peak 4
Blue    92.606351  54.209875   2.402143   1.375800
Green    6.225276  41.943145   2.091145   1.704395
Purple   0.643068   2.420364  89.907463  21.142949
Orange   0.525305   1.426616   5.599249  75.776855


## HBeta

In [24]:
mask_HB = (wv > l_HB-8)*(wv < l_HB+8)
image_HB = image_cut[:, mask_HB] 
image_error_HB = image_error_cut[:, mask_HB]
radial_profile_HB = np.sum(image_HB, axis = 1)
radial_profile_error_HB = np.sqrt(np.sum(image_error_HB**2, axis = 1))
fit = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/Mrk_544/FORS2.2021-08-21T05_32_16.048/Reduced/Fit_code/Hbeta_fit.csv", index_col = 0)

In [25]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 1'].iloc[3] -sigma, 
                                                                fit_HA['Component 1'].iloc[3] + sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 2'].iloc[3] -sigma, 
                                                                fit_HA['Component 2'].iloc[3] + sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 3'].iloc[3] -sigma, 
                                                                fit_HA['Component 3'].iloc[3] + sigma)
data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 4'].iloc[3] -sigma, 
                                                                fit_HA['Component 4'].iloc[3] + sigma)
data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(4)],
                  index=['Blue', 'Green', 'Purple', 'Orange'])
print(df)

df_path = os.path.join('Areas_Hbeta_1s.csv')
df.to_csv(df_path, float_format='%.3f')

6.655922699004513
6.452112690799256
1.244375985041566
1.2411442918429039
1.9066979957817431
2.2171396996579755
0.9395821137210687
0.9327918901173767
           Peak 1     Peak 2     Peak 3     Peak 4
Blue    87.772721  36.523462   2.038368   2.548011
Green   10.596923  56.003661   1.279874   0.519093
Purple   1.375955   6.332738  93.121550  26.131264
Orange   0.254401   1.140139   3.560208  70.801633


In [26]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 1'].iloc[3] -2*sigma, 
                                                                fit_HA['Component 1'].iloc[3] + 2*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 2'].iloc[3] -2*sigma, 
                                                                fit_HA['Component 2'].iloc[3] + 2*sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 3'].iloc[3] -2*sigma, 
                                                                fit_HA['Component 3'].iloc[3] + 2*sigma)
data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 4'].iloc[3] -2*sigma, 
                                                                fit_HA['Component 4'].iloc[3] + 2*sigma)
data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(4)],
                  index=['Blue', 'Green', 'Purple', 'Orange'])
print(df)

df_path = os.path.join('Areas_Hbeta_2s.csv')
df.to_csv(df_path, float_format='%.3f')

12.128106687733794
11.803314816261143
6.235729114163625
6.218185676509178
8.110478560345143
9.141997888928874
2.1846026470275377
2.174041990972653
           Peak 1     Peak 2     Peak 3     Peak 4
Blue    86.298899  39.113915   2.426540   2.645904
Green   11.904745  53.345725   1.554813   0.524723
Purple   1.516201   6.394824  91.779090  26.979245
Orange   0.280155   1.145536   4.239557  69.850128


In [27]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 1'].iloc[3] -3*sigma, 
                                                                fit_HA['Component 1'].iloc[3] + 3*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 2'].iloc[3] -3*sigma, 
                                                                fit_HA['Component 2'].iloc[3] + 3*sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 3'].iloc[3] -3*sigma, 
                                                                fit_HA['Component 3'].iloc[3] + 3*sigma)
data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 4'].iloc[3] -3*sigma, 
                                                                fit_HA['Component 4'].iloc[3] + 3*sigma)
data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(4)],
                  index=['Blue', 'Green', 'Purple', 'Orange'])
print(df)

df_path = os.path.join('Areas_Hbeta_3s.csv')
df.to_csv(df_path, float_format='%.3f')


18.04400141059794
17.6598927347682
8.810661811312668
8.782879614648905
10.002644458148842
11.073813817358689
2.936330726654161
2.9247873837182117
           Peak 1     Peak 2     Peak 3     Peak 4
Blue    81.412019  42.308563   2.793031   2.778841
Green   16.369285  50.127220   1.828916   0.564636
Purple   1.873010   6.420545  90.496413  28.665819
Orange   0.345686   1.143673   4.881640  67.990704


## NII

In [28]:
mask_NII = (wv > l_NII-8)*(wv < l_NII+8)
image_NII = image_cut[:, mask_NII]
image_error_NII = image_error_cut[:, mask_NII]
radial_profile_NII = np.sum(image_NII, axis = 1)
radial_profile_error_NII = np.sqrt(np.sum(image_error_NII**2, axis = 1))
fit = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/Mrk_544/FORS2.2021-08-21T05_32_16.048/Reduced/Fit_code/NII_fit.csv", index_col = 0)

In [29]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 1'].iloc[3] -sigma, 
                                                                fit_HA['Component 1'].iloc[3] + sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 2'].iloc[3] -sigma, 
                                                                fit_HA['Component 2'].iloc[3] + sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 3'].iloc[3] -sigma, 
                                                                fit_HA['Component 3'].iloc[3] + sigma)
data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 4'].iloc[3] -sigma, 
                                                                fit_HA['Component 4'].iloc[3] + sigma)
data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(4)],
                  index=['Blue', 'Green', 'Purple', 'Orange'])
print(df)

df_path = os.path.join('Areas_NII_1s.csv')
df.to_csv(df_path, float_format='%.3f')

5.130070519636471
5.30259398702816
1.4823697604250783
1.5128024903611643
2.685789089538906
3.1862673472169476
1.1815257831242236
1.1948743714867964
           Peak 1     Peak 2     Peak 3     Peak 4
Blue    88.596020  34.921669   1.656133   2.232892
Green    7.772401  55.063803   1.261273   1.145054
Purple   2.847600   8.047007  93.555554  32.237899
Orange   0.783980   1.967521   3.527040  64.384156


In [30]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 1'].iloc[3] -2*sigma, 
                                                                fit_HA['Component 1'].iloc[3] + 2*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 2'].iloc[3] -2*sigma, 
                                                                fit_HA['Component 2'].iloc[3] + 2*sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 3'].iloc[3] -2*sigma, 
                                                                fit_HA['Component 3'].iloc[3] + 2*sigma)
data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 4'].iloc[3] -2*sigma, 
                                                                fit_HA['Component 4'].iloc[3] + 2*sigma)
data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(4)],
                  index=['Blue', 'Green', 'Purple', 'Orange'])
print(df)

df_path = os.path.join('Areas_NII_2s.csv')
df.to_csv(df_path, float_format='%.3f')

9.54262242924326
9.81864463292017
7.117280164802876
7.21913820412071
11.384742469282983
13.057129926730534
2.8049397104130454
2.82956448365455
           Peak 1     Peak 2     Peak 3     Peak 4
Blue    87.472712  38.864252   1.979006   2.267932
Green    8.610914  50.611913   1.520823   1.152415
Purple   3.071911   8.466681  92.302505  32.664213
Orange   0.844463   2.057154   4.197666  63.915440


In [31]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 1'].iloc[3] -3*sigma, 
                                                                fit_HA['Component 1'].iloc[3] + 3*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 2'].iloc[3] -3*sigma, 
                                                                fit_HA['Component 2'].iloc[3] + 3*sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 3'].iloc[3] -3*sigma, 
                                                                fit_HA['Component 3'].iloc[3] + 3*sigma)
data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 4'].iloc[3] -3*sigma, 
                                                                fit_HA['Component 4'].iloc[3] + 3*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(4)],
                  index=['Blue', 'Green', 'Purple', 'Orange'])
print(df)

df_path = os.path.join('Areas_NII_3s.csv')
df.to_csv(df_path, float_format='%.3f')

14.77149543124625
15.09349794850898
9.789843780098495
9.90533498249635
13.998824697046874
15.739591923062992
3.798243360156581
3.8239110757193746
           Peak 1     Peak 2     Peak 3     Peak 4
Blue    83.435557  42.768082   2.285481   2.365543
Green   11.950692  46.413204   1.773777   1.209419
Purple   3.627320   8.715775  91.114188  34.406673
Orange   0.986432   2.102939   4.826554  62.018365


## SII

In [32]:
mask_SII = (wv > l_SII-8)*(wv < l_SII+8)
x_axes = np.arange(0, 100, 1)
image_SII = image_cut[:, mask_SII] 
image_error_SII = image_error_cut[:, mask_SII]
radial_profile_SII = np.sum(image_SII, axis = 1)
radial_profile_error_SII = np.sqrt(np.sum(image_error_SII**2, axis = 1))
fit = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/Mrk_544/FORS2.2021-08-21T05_32_16.048/Reduced/Fit_code/SII_fit.csv", index_col = 0)

In [33]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 1'].iloc[3] -sigma, 
                                                                fit_HA['Component 1'].iloc[3] + sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 2'].iloc[3] -sigma, 
                                                                fit_HA['Component 2'].iloc[3] + sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 3'].iloc[3] -sigma, 
                                                                fit_HA['Component 3'].iloc[3] + sigma)
data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 4'].iloc[3] -sigma, 
                                                                fit_HA['Component 4'].iloc[3] + sigma)
data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(4)],
                  index=['Blue', 'Green', 'Purple', 'Orange'])
print(df)

df_path = os.path.join('Areas_SII_1s.csv')
df.to_csv(df_path, float_format='%.3f')

2.5746410974896783
2.6362378985982797
0.860760952356287
0.8696676089303057
1.8616010804923977
2.0477189471092574
0.862750204046314
0.8596128247381571
           Peak 1     Peak 2     Peak 3     Peak 4
Blue    81.848233  36.842781   1.583401   1.921091
Green   13.413020  51.812371   3.024434   4.656505
Purple   3.155711   8.120209  91.281733  27.450356
Orange   1.583036   3.224640   4.110432  65.972048


In [34]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 1'].iloc[3] -2*sigma, 
                                                                fit_HA['Component 1'].iloc[3] + 2*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 2'].iloc[3] -2*sigma, 
                                                                fit_HA['Component 2'].iloc[3] + 2*sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 3'].iloc[3] -2*sigma, 
                                                                fit_HA['Component 3'].iloc[3] + 2*sigma)
data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 4'].iloc[3] -2*sigma, 
                                                                fit_HA['Component 4'].iloc[3] + 2*sigma)
data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(4)],
                  index=['Blue', 'Green', 'Purple', 'Orange'])
print(df)

df_path = os.path.join('Areas_SII_2s.csv')
df.to_csv(df_path, float_format='%.3f')

4.8871950490291285
4.98544304494113
4.201666041493932
4.23172962561018
7.918484110083157
8.531944040722898
2.0135873196438947
2.0093638097460564
           Peak 1     Peak 2     Peak 3     Peak 4
Blue    80.660192  39.715557   1.885818   1.980649
Green   14.331663  48.555495   3.590387   4.832953
Purple   3.338000   8.415972  89.654334  28.211728
Orange   1.670145   3.312975   4.869461  64.974670


In [35]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 1'].iloc[3] -3*sigma, 
                                                                fit_HA['Component 1'].iloc[3] + 3*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 2'].iloc[3] -3*sigma, 
                                                                fit_HA['Component 2'].iloc[3] + 3*sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 3'].iloc[3] -3*sigma, 
                                                                fit_HA['Component 3'].iloc[3] + 3*sigma)
data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 4'].iloc[3] -3*sigma, 
                                                                fit_HA['Component 4'].iloc[3] + 3*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(4)],
                  index=['Blue', 'Green', 'Purple', 'Orange'])
print(df)

df_path = os.path.join('Areas_SII_3s.csv')
df.to_csv(df_path, float_format='%.3f')

7.848137659538294
7.962251543441321
5.8259445061802975
5.860694291883754
9.765878641210927
10.40019713584062
2.720795679743661
2.7159496770675378
           Peak 1     Peak 2     Peak 3     Peak 4
Blue    76.783666  42.508192   2.171635   2.072120
Green   17.526785  45.527550   4.120060   5.039926
Purple   3.820557   8.608215  88.132972  29.853049
Orange   1.868992   3.356043   5.575333  63.034905


## OIII

In [36]:
mask_OIII = (wv > l_OIII-8)*(wv < l_OIII+8)
image_OIII = image_cut[:, mask_OIII] 
image_error_OIII = image_error_cut[:, mask_OIII]
radial_profile_OIII = np.sum(image_OIII, axis = 1)
radial_profile_error_OIII = np.sqrt(np.sum(image_error_OIII**2, axis = 1))
fit = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/Mrk_544/FORS2.2021-08-21T05_32_16.048/Reduced/Fit_code/OIII_fit.csv", index_col = 0)

In [37]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 1'].iloc[3] -sigma, 
                                                                fit_HA['Component 1'].iloc[3] + sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 2'].iloc[3] -sigma, 
                                                                fit_HA['Component 2'].iloc[3] + sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 3'].iloc[3] -sigma, 
                                                                fit_HA['Component 3'].iloc[3] + sigma)
data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 4'].iloc[3] -sigma, 
                                                                fit_HA['Component 4'].iloc[3] + sigma)
data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(4)],
                  index=['Blue', 'Green', 'Purple', 'Orange'])
print(df)

df_path = os.path.join('Areas_OIII_1s.csv')
df.to_csv(df_path, float_format='%.3f')

9.99846522549739
8.480604172623949
1.142242989174234
1.1436064950939808
2.0690154129672154
2.4543008172638823
1.0719924711288535
1.0770880107500869
           Peak 1     Peak 2     Peak 3     Peak 4
Blue    92.717031  57.951010   4.263348   6.767898
Green    5.759648  31.222732   0.047460   0.000018
Purple   1.084081   7.962170  90.482796  26.643347
Orange   0.439240   2.864088   5.206397  66.588737


In [38]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 1'].iloc[3] -2*sigma, 
                                                                fit_HA['Component 1'].iloc[3] + 2*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 2'].iloc[3] -2*sigma, 
                                                                fit_HA['Component 2'].iloc[3] + 2*sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 3'].iloc[3] -2*sigma, 
                                                                fit_HA['Component 3'].iloc[3] + 2*sigma)
data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 4'].iloc[3] -2*sigma, 
                                                                fit_HA['Component 4'].iloc[3] + 2*sigma)
data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(4)],
                  index=['Blue', 'Green', 'Purple', 'Orange'])
print(df)

df_path = os.path.join('Areas_OIII_2s.csv')
df.to_csv(df_path, float_format='%.3f')

17.72751000925801
15.320525643754399
5.891273451972188
5.893055400558196
8.877749274380776
10.163782289567159
2.576057642504723
2.5854943473740906
           Peak 1     Peak 2     Peak 3     Peak 4
Blue    91.811974  59.890357   5.009195   6.854917
Green    6.464248  29.516402   0.071855   0.000021
Purple   1.227309   7.804994  88.806119  26.642985
Orange   0.496469   2.788247   6.112831  66.502077


In [39]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 1'].iloc[3] -3*sigma, 
                                                                fit_HA['Component 1'].iloc[3] + 3*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 2'].iloc[3] -3*sigma, 
                                                                fit_HA['Component 2'].iloc[3] + 3*sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 3'].iloc[3] -3*sigma, 
                                                                fit_HA['Component 3'].iloc[3] + 3*sigma)
data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 4'].iloc[3] -3*sigma, 
                                                                fit_HA['Component 4'].iloc[3] + 3*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(4)],
                  index=['Blue', 'Green', 'Purple', 'Orange'])
print(df)

df_path = os.path.join('Areas_OIII_3s.csv')
df.to_csv(df_path, float_format='%.3f')

24.90579238344399
22.15487118821416
8.63100310965306
8.601756877950416
11.024598850632335
12.36298268934117
3.5099363636191048
3.5198564705886604
           Peak 1     Peak 2     Peak 3     Peak 4
Blue    89.329855  62.333318   5.698693   7.078256
Green    8.428058  27.439675   0.103692   0.000038
Purple   1.601305   7.550321  87.248961  27.902831
Orange   0.640782   2.676686   6.948654  65.018876
